In [146]:
import pdfplumber
import pandas as pd
import os
import re
from img2table.document import Image
from io import BytesIO
from pdf2image import convert_from_path
from img2table.ocr import TesseractOCR
from img2table.document import PDF
import json

In [147]:
ocr = TesseractOCR()

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.58 : libtiff 4.7.2 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.8 zlib/1.2.12 liblzma/5.8.3 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.3 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.64.0


In [148]:
def extract_headers(page):
    lines = page.extract_text_lines()
    return pd.DataFrame([line for line in lines if re.search('^[A-Z]{1}[0-9]{1,2}', line['text']) is not None])

In [149]:
def select_best_heading(page, table, headers):
    if len(headers) == 0:
        return ''
    headers['is_above'] = headers['top'].apply(lambda x: x < table.bbox.relative.y1 * page.height)
    headers = headers[headers['is_above']].sort_values('top', ascending=False).reset_index(drop=True)
    if len(headers) == 0:
        return ''
    return headers['text'][0]

In [150]:
def parse_page_tables(file, page_index):
    pdf = pdfplumber.open(file)
    page = pdf.pages[page_index]
    headers = extract_headers(page)

    doc = PDF(file,
          pages=[page_index],
          detect_rotation=False,
          pdf_text_extraction=True)

    results = doc.extract_tables(ocr=ocr,
                    implicit_rows=False,
                    implicit_columns=False,
                    borderless_tables=False,
                    min_confidence=50,
                    max_workers=1)

    data = []
    for table in results[page_index]:
        heading = select_best_heading(page, table, headers)
        code_match = re.search('^[A-Z]{1}[0-9]+', heading)
        if code_match is not None:
            code = code_match.group(0).strip()
        else:
            code = ''

        data.append({
            'table_code': code,
            'table_title': heading.replace(code, '').strip(),
            'table_records': table.df.fillna('').to_dict(orient='records'),
            'table_html': re.sub('[\n ]+', ' ', table.html)
        })

    return data

In [151]:
def extract_data_from_file(file):
    print(file)
    try:
        pdf = pdfplumber.open(file)
    except:
        return None
    unitid = file.split('/')[-1].replace('.pdf', '').strip()

    data = []
    for page in range(0, len(pdf.pages)):
        try:
            data.extend(parse_page_tables(file, page))
        except:
            continue

    if len(data) == 0:
        return None

    data = pd.DataFrame(data)

    data = data[data['table_code'] != '']

    data = data.groupby('table_code').agg({
        'table_code': first,
        'table_title': first,
        'table_records': list,
        'table_html': lambda tables: '<div class="separator"></div>'.join(tables)
    }).reset_index(drop=True)

    data['table_num'] = data['table_code'].apply(lambda x: re.search('[0-9]+', x).group(0)).apply(int)
    data['section'] = data['table_code'].apply(lambda x: re.search('[A-Z]+', x).group(0))
    data = data.sort_values(['section', 'table_num'])

    data = data[['section', 'table_num', 'table_code', 'table_title', 'table_records', 'table_html']]
    data['unitid'] = unitid

    return data

In [152]:
def first(lst):
    return list(lst)[0]

In [153]:
directory = pd.read_csv('../data/ipeds/hd2025.csv')
directory = directory[['UNITID', 'INSTNM', 'STABBR']]

In [154]:
years = os.listdir('../data/cds-docs')
year = '2024-2025'

In [155]:
files = [f'../data/cds-docs/{year}/{file}' for file in os.listdir(f'../data/cds-docs/{year}')]

In [164]:
df = pd.concat([extract_data_from_file(file) for file in files[0:3]]).reset_index(drop=True)

../data/cds-docs/2024-2025/195003.pdf
../data/cds-docs/2024-2025/110592.pdf
../data/cds-docs/2024-2025/155025.pdf


In [165]:
df['year'] = year

In [174]:
for unitid in df['unitid'].unique():
    college_df = df.query(f'unitid == "{unitid}"').reset_index(drop=True)
    for year in college_df['year'].unique():
        temp = college_df.query(f'year == "{year}"').reset_index(drop=True)

In [175]:
temp

,section,table_num,table_code,table_title,table_records,table_html,unitid,year
0,B,1,B1,. Institutional Enrollment - Men and Women,"[[{0: '', 1: 'FULL-TIME', 2: 'FULL-TIME', 3: '...","<table> <tr> <td rowspan=""2""> </td> <td colspa...",110592,2024-2025
1,B,2,B2,. Enrollment by Racial/Ethnic Category,"[[{0: 'Racial/Ethnic Category', 1: 'Degree-see...",<table> <tr> <td> Racial/Ethnic Category </td>...,110592,2024-2025
2,B,3,B3,. Persistence,"[[{0: 'AWARD TYPE', 1: '# AWARDED'}, {0: 'Cert...",<table> <tr> <td> AWARD TYPE </td> <td> # AWAR...,110592,2024-2025
3,B,4,B4,-B21: Graduation Rates,"[[{0: '', 1: '', 2: 'Recipients of a Federal G...","<table> <tr> <td colspan=""2""> </td> <td> Recip...",110592,2024-2025
4,C,1,C1,". First-time, first-year students: Provide the...","[[{0: 'FIRST-TIME, FIRST-YEAR STUDENT APPLICAN...","<table> <tr> <td> FIRST-TIME, FIRST-YEAR STUDE...",110592,2024-2025
5,C,2,C2,". First time, first-year wait-listed students","[[{0: 'WAITING LIST', 1: 'TOTAL'}, {0: 'Number...",<table> <tr> <td> WAITING LIST </td> <td> TOTA...,110592,2024-2025
6,C,5,C5,. Distribution of high school units required a...,"[[{0: '', 1: 'Units Required', 2: 'Units Recom...",<table> <tr> <td> </td> <td> Units Required </...,110592,2024-2025
7,C,9,C9,". Percent and number of first-time, first-year...","[[{0: 'Assessment', 1: '25th Percentile Score'...",<table> <tr> <td> Assessment </td> <td> 25th P...,110592,2024-2025
8,C,11,C11,". Percentage of all enrolled, degree-seeking, ...","[[{0: 'Score Range', 1: 'Percent (Students who...",<table> <tr> <td> Score Range </td> <td> Perce...,110592,2024-2025
9,C,12,C12,. Average high school GPA of all degree-seekin...,"[[{0: '', 1: '%'}, {0: 'Percent Submitting GPA...",<table> <tr> <td> </td> <td> % </td> </tr> <tr...,110592,2024-2025


In [171]:
df['unitid'].unique()

<StringArray>
['195003', '110592']
Length: 2, dtype: str

In [136]:
temp = df.query(f'unitid == "{unitid}"')

In [638]:
data = {
    unitid: data.to_dict(orient='records')
}

In [640]:
with open('../web-app/data.json', 'w') as out_file:
    json.dump(data, out_file)

In [641]:
# with open('test.html', 'w') as out_file:
#     out_file.write(results[4][0].html)